# Residual Analysis & Model Diagnostics Lab

A single metric like MAE, RMSE, or R² summarizes fit quality into one number, but hides critical patterns. Residuals reveal the true health of a model. This lab walks through diagnostic tests, residual normality checks, heteroscedasticity detection, and outlier impact on error metrics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from scipy import stats

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 1. Good Residuals (Linear Model on Linear Data)

In a well-specified model, residuals should be randomly scattered around zero with constant variance (homoscedasticity) and follow an approximately normal distribution.

In [ ]:
# Generate linear data
X = np.linspace(0, 10, 50).reshape(-1, 1)
y = 2 * X.ravel() + 1 + np.random.normal(0, 1, 50)

model_linear = LinearRegression().fit(X, y)
y_pred = model_linear.predict(X)
residuals = y - y_pred

print(f"Linear Model RMSE: {np.sqrt(np.mean(residuals**2)):.3f}")
print(f"Linear Model R²:   {model_linear.score(X, y):.3f}")

# Diagnostic tests
print(f"Mean Residual:      {np.mean(residuals):.4f} (expected ~0)")
print(f"Std Deviation:      {np.std(residuals):.3f}")
print(f"Skewness:           {stats.skew(residuals):.3f} (expected ~0)")
print(f"Kurtosis:           {stats.kurtosis(residuals):.3f} (expected ~0)")

stat, p_value = stats.shapiro(residuals)
print(f"Shapiro-Wilk test:  p={p_value:.3f} (p > 0.05 indicates normal residuals)")

## 2. Bad Residuals: Nonlinearity (Underfitting)

When a linear model is fit to nonlinear data, the residuals exhibit a systematic curve, and the normality assumption is violated.

In [ ]:
# Quadratic data
X_nonlin = np.linspace(0, 10, 50).reshape(-1, 1)
y_nonlin = X_nonlin.ravel()**2 + np.random.normal(0, 5, 50)

# 1. Underfitting with Linear Model
model_wrong = LinearRegression().fit(X_nonlin, y_nonlin)
residuals_wrong = y_nonlin - model_wrong.predict(X_nonlin)
stat_wrong, p_wrong = stats.shapiro(residuals_wrong)
print(f"Linear Model on Nonlinear Data - RMSE: {np.sqrt(np.mean(residuals_wrong**2)):.3f}, Shapiro p: {p_wrong:.4f}")

# 2. Correct Specification with PolynomialFeatures
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X_nonlin)
model_correct = LinearRegression().fit(X_poly, y_nonlin)
residuals_correct = y_nonlin - model_correct.predict(X_poly)
stat_correct, p_correct = stats.shapiro(residuals_correct)
print(f"Quadratic Model on Nonlinear Data - RMSE: {np.sqrt(np.mean(residuals_correct**2)):.3f}, Shapiro p: {p_correct:.4f}")

## 3. Bad Residuals: Heteroscedasticity (Funnel Pattern)

Heteroscedasticity occurs when the spread of the residuals varies across fitted values (e.g., error increases as predictions grow).

In [ ]:
# Variance increases linearly with X
X_het = np.linspace(0, 10, 50).reshape(-1, 1)
y_het = 2 * X_het.ravel() + 1 + np.random.normal(0, X_het.ravel() * 0.5, 50)

model_het = LinearRegression().fit(X_het, y_het)
y_pred_het = model_het.predict(X_het)
residuals_het = y_het - y_pred_het

# Analyze variance across fitted value bins
sort_idx = np.argsort(y_pred_het)
fitted_bins = np.array_split(y_pred_het[sort_idx], 5)
residual_bins = np.array_split(residuals_het[sort_idx], 5)

print(f"{'Bin':<6} {'Fitted Range':<18} {'Residual Std Dev':<15}")
print("-" * 42)
for i, (f_bin, r_bin) in enumerate(zip(fitted_bins, residual_bins)):
    print(f"{i+1:<6} [{f_bin.min():.1f}, {f_bin.max():.1f}]       {np.std(r_bin):.3f}")

## 4. Metric Comparison: MAE vs. RMSE Outlier Sensitivity

See how a single large outlier influences MAE, RMSE, and R².

In [ ]:
# Normal dataset
X_clean = np.linspace(0, 10, 20).reshape(-1, 1)
y_clean = 2 * X_clean.ravel() + 1 + np.random.normal(0, 1, 20)

# Dataset corrupted by a single outlier
y_corrupt = y_clean.copy()
y_corrupt[-1] = 50.0

def eval_metrics(X, y):
    m = LinearRegression().fit(X, y)
    preds = m.predict(X)
    mae = np.mean(np.abs(y - preds))
    rmse = np.sqrt(np.mean((y - preds)**2))
    r2 = m.score(X, y)
    return mae, rmse, r2

mae_clean, rmse_clean, r2_clean = eval_metrics(X_clean, y_clean)
mae_bad, rmse_bad, r2_bad = eval_metrics(X_clean, y_corrupt)

print(f"{'Metric':<8} {'Clean Data':<12} {'With 1 Outlier':<15} {'Absolute Delta':<15}")
print("-" * 50)
print(f"{'MAE':<8} {mae_clean:<12.3f} {mae_bad:<15.3f} {mae_bad - mae_clean:+.3f}")
print(f"{'RMSE':<8} {rmse_clean:<12.3f} {rmse_bad:<15.3f} {rmse_bad - rmse_clean:+.3f}")
print(f"{'R²':<8} {r2_clean:<12.3f} {r2_bad:<15.3f} {r2_bad - r2_clean:+.3f}")